# Post5G - Simple Scenario

## Prerequisites

Before starting, make sure you:

- Have completed the [Preparation](./00_preparation.ipynb), [Provisioning Resources](./01_provision.ipynb), and [Kubernetes Cluster](./02_k8s.ipynb) tutorials.
- Have a basic understanding of Python.
- Are familiar with 5G architecture.

## Tutorial Overview

This scenario introduces you to the orchestrator by deploying a complete, fully virtualized 5G end-to-end network.

![Simple topology](figures/simple.png)

### What You Will Deploy

The network architecture consists of:
* **5G Core:** Implemented using **Open5GS**, deployed on a virtual machine (`node-core`).
* **gNB:** Implemented using **UERANSIM** (`gnb`), deployed on a virtual machine (`node-ran`).
* **User Equipment (UE):** Two simulated UEs (`sim_ue1` and `sim_ue2`) implemented via **UERANSIM**, deployed on a virtual machine (`node-ues`).

Because this network is fully virtualized with the core and RAN emulated and the radio channel/UEs simulated—there is no need for a physical radio unit or specific hardware. 

### UE Configuration

The network utilizes a single Data Network Name (DNN) (`internet`) and one Single-Network Slice Selection Assistance Information (S-NSSAI) (`internet`). The UEs are configured with the following parameters:

| UE | SUPI | S-NSSAI | DNN |
| :--- | :--- | :--- | :--- |
| `sim_ue1` | `001010000000100` | `internet` | `internet` |
| `sim_ue2` | `001010000000101` | `internet` | `internet` |

### Target Infrastructure

To run this scenario, we will use ephemeral virtual machines provided by the [SLICES Basic Infrastructure Service (SLICES BI)](https://doc.slices-ri.eu/BasicServices/BasicInfrastructure/basicinfrastructure.html). However, you can also adapt this deployment to use your own virtual machines or bare-metal servers as shown in the [Provisioning Resources tutorial](./01_provision.ipynb).

## Metamodel and Deployement model definition

### Metamodel
The Post-5G blueprint abstracts the 5G network with a generic **metamodel**
compliant with 3GPP specifications. The metamodel is essentially a
[Pydantic model](https://pydantic.dev/docs/validation/dev/get-started/) that
helps to define the settings of the 5G network to operate in a declarative way.

To simplify the usage of the metamodel, it comes with a full set of default
values. That way, the user only needs to specify the settings for elements that
deviate from the default values.

In this example we follow a leaning-by-doing approach, but you can visit the
[API Reference](https://api-reference-17fc2f.gitlabpages.inria.fr)
to obtain the full specifications and default values.

In our case, we have to define a core, one gNB, and two UEs as stated above.

In this example, we will deploy a core with the default settings so we don't have
to explicitly mention and configure it.

We however have to declare the gNB and the UEs as described above. To that aim,
we simply have to instanciate an object of the `MetaModel` class and explicitely
populate it with our desired elements.

The following snippet defines the desired RAN and UEs.


In [1]:
from metamodel.models.metamodel import MetaModel

metamodel=MetaModel.from_yaml(
"""
ran:
    components:
        gnb:
            type: gnb
ues:
  sim_ue1:
    config:
      supi: '001010000000100'
      subscriptions:
      - snssai: internet
        dnn: internet
      gnbs:
      - gnb
  sim_ue2:
    config:
      supi: '001010000000101'
      subscriptions:
      - snssai: internet
        dnn: internet
      gnbs:
      - gnb
""")

**NOTE:** Because the radio channel is simulated, you must explicitly specify the gNBs to which the UEs connect.

### Understanding the Deployment Model

A key feature of the metamodel is that it is **implementation-agnostic**. This allows you to share 5G configurations and run experiments across various environments and implementations.

The orchestrator library instantiates these metamodels for specific environments by pairing them with a **deployment model**, which contains the configuration details required for a target deployment. The deployment model specifies:
* Which implementations to use for each 5G function.
* Where and how to deploy them.

Like the metamodel, the deployment model is a Pydantic model with built-in default values to simplify configuration. To create one, simply instantiate the `DeploymentModel` class and configure it to match your requirements.

### Configuring Functions and Target Nodes

By default, the core network uses Open5GS within our orchestrator, so no explicit configuration is required. However, you must explicitly declare that the gNB and UEs use **UERANSIM** by defining them under the `ran` and `ues` sections.

You also need to declare the target nodes for your functions. In the `nodes` section, you list these machines and tag them with the roles they support:
* **`node-core`:** Tagged with `core` and `ric` to run the core network and RIC.
* **`node-ran`:** Tagged with `ran` to accept RAN components.
* **`node-ues`:** Tagged with `ue` to host UE instances.

### Provisioning Resources with SLICES BI

To run this scenario using ephemeral virtual machines on the **SLICES BI** infrastructure, the orchestrator utilizes a collection of built-in provisioners (see the [Provisioning Resources](./01_provision.ipynb) tutorial). 

In this example, we use a provisioner named `my-provisionner` to automatically spawn and configure VMs, which is why each defined machine includes the `provisioner: my-provisionner` attribute.

Each provisioner accepts specific configuration parameters. For this tutorial:
* **Infrastructure:** `fr-sophia2-bi-vm1`
* **Project:** `post5g-beta`
* **Experiment Name:** `simple_scenario` (automatically created by the provisioner)
* **Node Lifetime:** 8 hours (after which they are automatically destroyed)

> **Important:** Make sure your public SSH keys are registered to SLICES before deploying your network. If you need a refresher on these concepts, check the [preparation tutorial](./00_preparation.ipynb).

> **Note:** Components are referenced by name in both the metamodel and deployment model. For example, `provisioner: my-provisionner` tells the orchestrator to use the provisioner explicitly named `my-provisionner`. This name-based referencing allows you to swap out provisioner types (e.g., switching from virtual machines to bare-metal servers or onboarded Linux hosts) without changing your core node definitions—making it ideal for exploring different parameter spaces.

In [2]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
ran:
  components:
    gnb:
      type: gnb
      implementation: ueransim
ues:
  sim_ue1:
    implementation: ueransim
    type: containerized
  sim_ue2:
    implementation: ueransim
    type: containerized
nodes:
  node-core:
    provisioner: my-provisionner
    roles:
    - core
    - ric
  node-ran:
    provisioner: my-provisionner
    roles:
    - ran
  node-ues:
    provisioner: my-provisionner
    roles:
    - ue
provisioners:
  my-provisionner:
    type: bi
    params:
      infra: be-gent1-bi-vm1
      project: post5g-beta
      experiment: simple_scenario
      create_experiment: true
      experiment_duration: 1h
      node_duration: 1h
"""
)



## Provisioning the Infrastructure

### Create Ephemeral Virtual Machines

Now that we have defined both the metamodel and deployment model, we can trigger resource provisioning using the appropriate **Manager**. 

Managers in the orchestrator are designed for specific operational tasks. The **`ProvisioningManagement`** manager handles provisioning the machines listed in the `nodes` section of the deployment model, automatically selecting the correct provisioner for each node.

1. **Initialize the Manager:** First, instantiate the `ProvisioningManagement` manager by passing in your metamodel and deployment model. This stage does not make changes to the infrastructure; it validates and reconciles the models to ensure consistency.
2. **Trigger Provisioning:** Once validated, execute the provisioning command to spawn and configure the actual target nodes.

In [3]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)
await prov.provision(quiet=False)

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' for 'node-ran' provisioning] ***

TASK [provisioning/bi/provision : Select project 'post5g-beta' for 'node-core' provisioning] ***

PLAY [localhost] ********************************

We now have our three virtual machines deployed in SLICES BI and configured according to the requirements of this scenario.

> **Optional:** You can verify the deployment by listing the experiment's active resources using the SLICES CLI (be sure to adapt the parameters to match your setup):

In [4]:
!SLICES_BI_SITE_ID=be-gent1-bi-vm1 slices bi list-resources --experiment simple_scenario

  Resources in experiment exp_expauth.ilabt.imec.be_01m6km6ev3efmse0kdt115qqdf  
┏━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┓
┃           ┃           ┃        ┃ Created   ┃ Expires  ┃ Public    ┃ Private  ┃
┃ ID        ┃ Name      ┃ Status ┃ At        ┃ At       ┃ IPv4      ┃ IPv4     ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━┩
│ r_be-gen… │ node-ues  │ up     │ 2026-11-… │ 2026-11… │           │ 10.10.2… │
│           │           │        │ 15:52 UTC │ 16:52    │           │          │
│           │           │        │           │ UTC      │           │          │
│ r_be-gen… │ node-ran  │ up     │ 2026-11-… │ 2026-11… │           │ 10.10.2… │
│           │           │        │ 15:52 UTC │ 16:52    │           │          │
│           │           │        │           │ UTC      │           │          │
│ r_be-gen… │ node-core │ up     │ 2026-11-… │ 2026-11… │           │ 10.10.2… │
│           │           │   

Each manager accepts a metamodel and a deployment model as input, and outputs an updated deployment model containing the results of its actions. This updated model can then be passed directly to the next manager in the workflow chain.

Several methods are available to inspect and extract these models:
* **`output()` Method:** Directly returns both the metamodel and the updated deployment model. 

As expected, the resulting deployment model includes all necessary access details (such as IP addresses and credentials) for the newly provisioned machines.

In [5]:
metamodel, deployment_model = prov.output()
print (deployment_model.to_yaml())

provisioners:
  my-provisionner:
    type: bi
    params:
      infra: be-gent1-bi-vm1
      experiment: simple_scenario
      create_experiment: true
      experiment_duration: 1h
      node_duration: 1h
nodes:
  node-core:
    name: node-core
    is_provisioned: true
    provisioner: my-provisionner
    ssh_info:
      hostname: 10.10.222.15
      user: ubuntu
      jumphost:
        hostname: bastion2.slices-be.eu
        user: proxy
        common_args: -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o
          ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o
          ControlPersist=7200s -o ControlPath=/tmp/cm-%C
      common_args: -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o
        ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o ControlPersist=7200s
        -o ControlPath=/tmp/cm-%C -o ProxyCommand="ssh -o StrictHostKeyChecking=no
        -o UserKnownHostsFile=/dev/null -W %h:%p proxy@bastion2.slic

It can also be useful to save these models to files—for instance, to resume your work later or share the configuration across different workflow steps (we demonstrate this pattern later in the tutorial).

In [6]:
from pathlib import Path
DIR="state/simple"
Path(DIR).mkdir(parents=True, exist_ok=True)

prov.metamodel.to_yaml_file(f"{DIR}/metamodel.prov.yaml")
prov.deployment_model.to_yaml_file(f"{DIR}/deployment_model.prov.yaml")

### Build the Kubernetes Cluster

The orchestrator relies on Kubernetes to deploy most of its 5G network functions. The next step is to create a Kubernetes cluster using the virtual machines provisioned in the previous step.

To accomplish this, we use the **`K8SManagement`** manager. Its role is to deploy and configure a Kubernetes cluster tailored to our target 5G network layout. In this scenario, the cluster will include `node-core`, `node-ran`, and `node-ues`, each labeled so that workloads are scheduled onto the correct machines (for example, UEs will only be scheduled on `node-ues`).

#### Manager Chaining and Immutability

Because the orchestrator follows a declarative and functional paradigm:
* **Automatic Model Inheritance:** Managers consume and output metamodels and deployment models. When instantiating the Kubernetes manager, simply specify the preceding manager (e.g., `prov`), and it will automatically inherit the updated models containing the newly provisioned machine details.
* **Immutability:** Metamodels and deployment models are copied when passed between managers. Input models are **never mutated** in-place by a manager.

#### Cluster Deployment

Once the manager is instantiated, execute the cluster deployment command. This provisions the Kubernetes cluster across all specified nodes and installs a valid `kubeconfig` file on each machine to simplify subsequent management tasks.

> **Note:** To onboard an existing Kubernetes cluster rather than provisioning a new one, please refer to the [API Reference](https://api-reference-17fc2f.gitlabpages.inria.fr).

In [7]:
from orchestrator.deployment import K8SManagement

k8s = K8SManagement(preceding=prov)
await k8s.deploy(quiet=True)            # don't show output

At this stage, the infrastructure is fully ready to support our 5G network
deployment.

But before deploying the 5G network, let's archive the metamodel and deployment
model resulting from this deployment.

In [8]:
k8s.metamodel.to_yaml_file(f"{DIR}/metamodel.k8s.yaml")
k8s.deployment_model.to_yaml_file(f"{DIR}/deployment_model.k8s.yaml")

## Deploy the 5G Network

### Deploy the 5G Core

Now that the underlying infrastructure is fully provisioned, the first step is to deploy the 5G core network.

Deploying the core is straightforward: instantiate a **`CoreManagement`** manager and execute the deployment. Because all 5G-specific settings and deployment details are already defined in your metamodel and deployment model, the manager has everything it needs. As with previous stages, simply pass in the output from the preceding manager (i.e., `k8s`) to inherit the models automatically.

Once deployed, the core not only instantiates all required 5G network functions but also automatically registers the authentication and subscriber information for the UEs described in your metamodel into the User Data Repository (UDR).

In [9]:
from orchestrator.deployment import CoreManagement
core = CoreManagement(preceding=k8s)

In [11]:
await core.deploy()


PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [database/subscriber/add/open5gs : Add UE 'sim_ue1' to Open5GS database] ***

TASK [database/subscriber/add/open5gs : Add UE 'sim_ue2' to Open5GS database] ***
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node-core]
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
chang

By default, the Open5GS core is deployed in the `open5gs-core` namespace within the Kubernetes cluster, with its network functions restricted to nodes tagged with the `core` role.

#### Verifying the Core Deployment

To verify that the core functions are running correctly, you can list the active pods in that namespace. 

The **`ProvisioningManagement`** manager provides a simple yet powerful `run_node_command` function to execute shell commands directly on nodes. For example, you can run `kubectl get pods` on the `node-core` machine to check that all pods are deployed as expected.

> **Note:** Because the orchestrator follows a strict functional and immutable paradigm, managers never modify existing models in place. To run commands against the latest infrastructure state, you must instantiate a new **`ProvisioningManagement`** manager and import the models from the core deployment manager you just executed. An older `ProvisioningManagement` instance will still hold a stale deployment model that lacks the latest updates.

In [12]:
prov = ProvisioningManagement(preceding=core)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "open5gs-core", "-o", "wide"],
    nodes=["node-core"],
    become=True, quiet=True)))

NAME                            READY   STATUS    RESTARTS   AGE     IP               NODE        NOMINATED NODE   READINESS GATES
open5gs-amf-7b6dcfbfdf-mwfjz    1/1     Running   0          2m37s   10.244.134.7     node-core   <none>           <none>
open5gs-ausf-5778d7548b-59rww   1/1     Running   0          2m22s   10.244.134.8     node-core   <none>           <none>
open5gs-bsf-75cf9d7fb4-wnbpz    1/1     Running   0          53s     10.244.134.14    node-core   <none>           <none>
open5gs-database-0              1/1     Running   0          3m19s   10.244.134.5     node-core   <none>           <none>
open5gs-nrf-65d98494d7-9vkhr    1/1     Running   0          2m56s   10.244.134.6     node-core   <none>           <none>
open5gs-nssf-84cb8d64f7-6dj96   1/1     Running   0          68s     10.244.134.13    node-core   <none>           <none>
open5gs-pcf-6f444b66c5-t5qwv    1/1     Running   0          113s    10.244.134.10    node-core   <none>           <none>
open5gs-smf-5c7

In a very similar way, the logs of 5G functions can be retrieved directly with
the managers. For example, the logs of the AMF shows that it is well registered
to the NRF.

In [13]:
print(await core.get_logs(components=["amf"], quiet=True))

Open5GS daemon v2.8.0

11/04 15:58:48.513: [app] INFO: Configuration: '/opt/open5gs/etc/open5gs/config.yaml' (../lib/app/ogs-init.c:144)
11/04 15:58:48.513: [app] INFO: File Logging: '/opt/open5gs/var/log/open5gs/amf.log' (../lib/app/ogs-init.c:147)
11/04 15:58:48.513: [app] INFO: LOG-LEVEL: 'info' (../lib/app/ogs-init.c:150)
11/04 15:58:48.516: [sbi] INFO: Setup NF EndPoint(fqdn) [open5gs-nrf:0] (../lib/sbi/context.c:451)
11/04 15:58:48.517: [metrics] INFO: metrics_server() [http://10.244.134.7]:9090 (../lib/metrics/prometheus/context.c:392)
11/04 15:58:48.517: [sbi] INFO: NF Service [namf-comm] (../lib/sbi/context.c:2012)
11/04 15:58:48.517: [sbi] INFO: nghttp2_server() [http://10.244.134.7]:80 (../lib/sbi/nghttp2-server.c:454)
11/04 15:58:48.535: [amf] INFO: ngap_server() [10.244.134.7]:38412 (../src/amf/ngap-sctp.c:61)
11/04 15:58:48.535: [sctp] INFO: AMF initialize...done (../src/amf/app.c:33)
11/04 15:58:48.538: [sbi] INFO: [9f3fdc38-d868-41f1-9ea9-c798fddb4b17] NF registered [He

### Deploy the 5G RAN

Now that the 5G core is fully functional, you can deploy the Radio Access Network (RAN) using the **`RANManagement`** manager. 

Instantiate a `RANManagement` manager, import the metamodel and deployment model from the preceding deployment stage (i.e., `core`), and then trigger the deployment by calling the manager's `deploy()` method.

In [14]:
from orchestrator.deployment import RANManagement
ran = RANManagement(preceding=core)

/workspaces/orchestrator/.venv-1/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='snssaiList', input_value=Slice(sst=1, sd=None, nam...maximumBitRate=None)])}), input_type=Slice])
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='ru', input_value=RU(type=<RUType.SIMULATED...xGainDB=35, rxGainDB=75), input_type=RU])
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='amf', input_value=IPv4Address('10.244.134.7'), input_type=IPv4Address])
  return self.__pydantic_serializer__.to_python(


In [15]:
await ran.deploy()


PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'ueransim-gnb' chart] *****
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node-core]

TASK [helm/prepare : Copy custom Helm chart directory for 'ueransim-gnb' if provided] ***
skipping: [node-core]

TASK [helm/prepare : Copy local built-in 'ueransim-gnb' Helm chart directory for 'ueransim-gnb'] ***
changed: [node-core]

TASK [helm/prepare : Render built-in Helm Chart.yaml templated file for 'ueransim-gnb'] ***
changed: [node-core]

TASK [helm/prepare : Render custom templated Helm files for 'ueransim-gnb' if provided] ***
skipping: [node-core]

TASK [helm/prepare : Render

In this scenario, the RAN is configured with a single UERANSIM gNB named `gnb`. By default, the UERANSIM RAN is deployed in the `ueransim-ran` namespace. Following deployment, you should see a single active pod scheduled on the `node-ran` node.

Just as with the core network, you can verify this deployment by instantiating a new **`ProvisioningManagement`** manager and running a `kubectl` command to list the pods in the RAN namespace.

In [16]:
prov = ProvisioningManagement(preceding=ran)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "ueransim-ran", "-o", "wide"],
    nodes=["node-ran"],
    become=True, quiet=True)))

NAME                           READY   STATUS    RESTARTS   AGE   IP               NODE       NOMINATED NODE   READINESS GATES
ueransim-gnb-fdb8dd846-78k6j   1/1     Running   0          11s   10.244.137.250   node-ran   <none>           <none>


The output of the command confirms the proper deployment of the gNB on the
`node-ran` machine.

To confirm that the gNB is implemented with UERANSIM and is registered to the
core, we can analyse the logs of the gNB.

In [17]:
print(await ran.get_logs(components=["gnb"], quiet=True))

UERANSIM v3.3.0
[2026-11-04 16:02:04.625] [sctp] [info] Trying to establish SCTP connection... (10.244.134.7:38412)
[2026-11-04 16:02:04.646] [sctp] [info] SCTP connection established (10.244.134.7:38412)
[2026-11-04 16:02:04.646] [sctp] [debug] SCTP association setup ascId[3]
[2026-11-04 16:02:04.646] [ngap] [debug] Sending NG Setup Request
[2026-11-04 16:02:04.652] [ngap] [debug] NG Setup Response received
[2026-11-04 16:02:04.652] [ngap] [info] NG Setup procedure is successful



From the logs, we can confirm that the NG Setup procedure with the AMF was successful. Correlating these logs with the core pod listing shows that the AMF address used by the gNB correctly matches the AMF address of the core deployed earlier. 

Crucially, this IP address configuration is handled transparently by the orchestrator library, freeing you from manual networking overhead.

## Deploy the UEs

Now that both the 5G core network and RAN are fully functional, you can deploy the UEs defined in your metamodel.

To do so, instantiate a **`UEManagement`** manager, passing in the metamodel and deployment model from the previous stage (i.e., `ran`). Then, trigger the UE deployment by calling the manager's `deploy()` method.

When using UERANSIM, UEs are deployed into the `ueransim-ue` namespace by default. As in previous steps, you can run a `kubectl` command via a `ProvisioningManagement` manager to confirm that both UEs are running successfully.

In [18]:
from orchestrator.deployment import UEManagement
ues = UEManagement(preceding=ran)

In [19]:
await ues.deploy()


PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'ueransim-sim-ue2' chart] ***

TASK [helm/prepare : Create Helm build directory for 'ueransim-sim-ue1' chart] ***
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node-core]
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
cha

In [20]:
prov = ProvisioningManagement(preceding=ues)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "ueransim-ue", "-o", "wide"],
    nodes=["node-ues"],
    become=True, quiet=True)))

NAME                                READY   STATUS    RESTARTS   AGE   IP             NODE       NOMINATED NODE   READINESS GATES
ueransim-sim-ue1-6db84b5878-882cw   1/1     Running   0          9s    10.244.198.3   node-ues   <none>           <none>
ueransim-sim-ue2-6485fd4bd6-fx2d8   1/1     Running   0          9s    10.244.198.2   node-ues   <none>           <none>


As for the gNB or the AMF we can also analyse the logs the the UEs and see that
they both successuflly registered to the network and established a PDU session.

In [21]:
print(await ues.get_logs(components=["sim_ue1"], quiet=True))

UERANSIM v3.3.0
[2026-11-04 16:03:54.620] [nas] [info] UE switches to state [MM-DEREGISTERED/PLMN-SEARCH]
[2026-11-04 16:03:54.621] [rrc] [debug] New signal detected for cell[1], total [1] cells in coverage
[2026-11-04 16:03:54.621] [nas] [info] Selected plmn[001/01]
[2026-11-04 16:03:54.621] [rrc] [info] Selected cell plmn[001/01] tac[1] category[SUITABLE]
[2026-11-04 16:03:54.621] [nas] [info] UE switches to state [MM-DEREGISTERED/PS]
[2026-11-04 16:03:54.621] [nas] [info] UE switches to state [MM-DEREGISTERED/NORMAL-SERVICE]
[2026-11-04 16:03:54.621] [nas] [debug] Initial registration required due to [MM-DEREG-NORMAL-SERVICE]
[2026-11-04 16:03:54.621] [nas] [debug] UAC access attempt is allowed for identity[0], category[MO_sig]
[2026-11-04 16:03:54.621] [nas] [debug] Sending Initial Registration
[2026-11-04 16:03:54.621] [nas] [info] UE switches to state [MM-REGISTER-INITIATED]
[2026-11-04 16:03:54.621] [rrc] [debug] Sending RRC Setup Request
[2026-11-04 16:03:54.622] [rrc] [info] R

In [22]:
print(await ues.get_logs(components=["sim_ue2"], quiet=True))

UERANSIM v3.3.0
[2026-11-04 16:03:54.045] [nas] [info] UE switches to state [MM-DEREGISTERED/PLMN-SEARCH]
[2026-11-04 16:03:54.045] [rrc] [debug] New signal detected for cell[1], total [1] cells in coverage
[2026-11-04 16:03:54.045] [nas] [info] Selected plmn[001/01]
[2026-11-04 16:03:54.045] [rrc] [info] Selected cell plmn[001/01] tac[1] category[SUITABLE]
[2026-11-04 16:03:54.045] [nas] [info] UE switches to state [MM-DEREGISTERED/PS]
[2026-11-04 16:03:54.046] [nas] [info] UE switches to state [MM-DEREGISTERED/NORMAL-SERVICE]
[2026-11-04 16:03:54.046] [nas] [debug] Initial registration required due to [MM-DEREG-NORMAL-SERVICE]
[2026-11-04 16:03:54.046] [nas] [debug] UAC access attempt is allowed for identity[0], category[MO_sig]
[2026-11-04 16:03:54.046] [nas] [debug] Sending Initial Registration
[2026-11-04 16:03:54.046] [nas] [info] UE switches to state [MM-REGISTER-INITIATED]
[2026-11-04 16:03:54.046] [rrc] [debug] Sending RRC Setup Request
[2026-11-04 16:03:54.046] [rrc] [info] R

To test that the UEs are properly attached and that the network functions as expected, you can send ICMP ping requests from the UEs over their 5G radio interface. When using UERANSIM, this virtual interface is named `uesimtun0`.

To run commands directly inside the UE containers on the node, use the `run_node_command` function provided by the `ProvisioningManagement` manager to execute a `kubectl exec` command against the target UE pod.

In [23]:
command = ["ping", "-I", "uesimtun0", "-c", 10, "-D", "8.8.8.8"]
pings = await ues.run_command(command=command, components=["sim_ue1", "sim_ue2"], quiet=True, flatten=False)

ping from `sim_ue1`:

In [24]:
print (pings["sim_ue1"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.3 uesimtun0: 56(84) bytes of data.
[1793808249.702698] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=117 time=5.12 ms
[1793808250.703568] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=117 time=4.80 ms
[1793808251.705629] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=117 time=4.94 ms
[1793808252.707620] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=117 time=4.92 ms
[1793808253.709535] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=117 time=4.85 ms
[1793808254.711600] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=117 time=4.98 ms
[1793808255.712715] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=117 time=5.03 ms
[1793808256.713599] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=117 time=4.79 ms
[1793808257.715308] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=117 time=4.72 ms
[1793808258.717206] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=117 time=4.83 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9015ms
rtt min/avg/max/mdev = 4.718/4.897/5.116/0.116 ms



ping from `sim_ue2`:

In [25]:
print (pings["sim_ue2"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.2 uesimtun0: 56(84) bytes of data.
[1793808249.702953] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=117 time=4.27 ms
[1793808250.704386] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=117 time=4.36 ms
[1793808251.705628] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=117 time=4.28 ms
[1793808252.707595] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=117 time=4.90 ms
[1793808253.709525] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=117 time=4.85 ms
[1793808254.711571] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=117 time=4.95 ms
[1793808255.712721] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=117 time=5.05 ms
[1793808256.713605] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=117 time=4.80 ms
[1793808257.715308] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=117 time=4.63 ms
[1793808258.717203] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=117 time=4.82 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9014ms
rtt min/avg/max/mdev = 4.268/4.691/5.052/0.275 ms



Finally, we can archive the metamodel and deployment model in file for keeping
track of the experiment models.

In [26]:
ues.metamodel.to_yaml_file(f"{DIR}/metamodel.ues.yaml")
ues.deployment_model.to_yaml_file(f"{DIR}/deployment_model.ues.yaml")